In [1]:
import os
import time
import requests
import pandas as pd
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(".env")
TMDB_TOKEN = os.getenv("TMDB_TOKEN")

OUTPUT_FILE = "Q6.csv"

tmdb_headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_TOKEN}"
}

BASE_URL = "https://api.themoviedb.org/3"

## Getter

In [2]:
def tmdb_get(endpoint, params=None, retries=3):
    if params is None:
        params = {}
    url = BASE_URL + endpoint
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=tmdb_headers, params=params, timeout=10)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.Timeout:
            print(f"Timeout, retry {attempt + 1}/{retries}...")
            time.sleep(2)
        except requests.exceptions.RequestException as e:
            print(f"Fehler: {e}")
            time.sleep(2)
    return None

## Collect Movies

In [3]:
def collect_movies(year, pages=30):
    all_movies = []
    for page in range(1, pages + 1):
        data = tmdb_get(
            "/discover/movie",
            params={
                "primary_release_year": year,
                "sort_by":              "popularity.desc",
                #"vote_count.gte":       100,
                "page":                 page,
            }
        )
        all_movies.extend(data["results"])
        time.sleep(0.25)
    return pd.DataFrame(all_movies)

## Load Movies

In [4]:
frames = []

for year in range(2000, 2025):
    frames.append(collect_movies(year, pages=30))
    print(f"Jahr {year} geladen")

df_movies = pd.concat(frames, ignore_index=True).drop_duplicates("id")
print(f"Gesamt: {len(df_movies)} Filme")

KeyboardInterrupt: 

## Load Details

In [ ]:
rows = []

for _, movie in df_movies.iterrows():
    details = tmdb_get(f"/movie/{movie['id']}")
    
    if details is None:
        continue
    
    if not details.get("belongs_to_collection"):
        continue
    
    budget  = details.get("budget",  0)
    revenue = details.get("revenue", 0)
    
    if budget == 0 or revenue == 0:
        continue
    
    rows.append({
        "collection_id":   details["belongs_to_collection"]["id"],
        "collection_name": details["belongs_to_collection"]["name"],
        "title":           details["title"],
        "release_date":    details["release_date"],
        "budget":          budget,
        "revenue":         revenue,
        "roi":             round(revenue / budget, 2),
        "rating":          details["vote_average"],
        "vote_count":      details["vote_count"],
    })
    
    time.sleep(0.25)

print(f"Franchise-Filme: {len(rows)}")

Timeout, retry 1/3...
Franchise-Filme: 712


## Build Dataframe

In [ ]:
df = pd.DataFrame(rows)
df["release_date"] = pd.to_datetime(df["release_date"])
df = df.sort_values(["collection_id", "release_date"]).reset_index(drop=True)

df["part"] = df.groupby("collection_id").cumcount()

counts = df.groupby("collection_id")["title"].transform("count")
df = df[counts >= 2].reset_index(drop=True)

print(f"Franchises: {df['collection_id'].nunique()}")
print(f"Filme gesamt: {len(df)}")

Franchises: 162
Filme gesamt: 414


## Compaire

In [ ]:
originals = df[df["part"] == 0][["collection_id", "revenue", "roi", "rating"]].rename(columns={
    "revenue": "orig_revenue",
    "roi":     "orig_roi",
    "rating":  "orig_rating"
})

df = df.merge(originals, on="collection_id", how="left")

df.loc[df["part"] == 0, ["orig_revenue", "orig_roi", "orig_rating"]] = None

df["revenue_diff"] = df["revenue"] - df["orig_revenue"]
df["roi_diff"]     = df["roi"]     - df["orig_roi"]
df["rating_diff"]  = df["rating"]  - df["orig_rating"]

df.head(10)

,collection_id,collection_name,title,release_date,budget,revenue,roi,rating,vote_count,part,orig_revenue,orig_roi,orig_rating,revenue_diff,roi_diff,rating_diff
0,10,Star Wars Collection,Star Wars: Episode II - Attack of the Clones,2002-05-15,120000000,649398328,5.41,6.589,14142,0,NaN,NaN,NaN,NaN,NaN,NaN
1,10,Star Wars Collection,Star Wars: Episode III - Revenge of the Sith,2005-05-17,113000000,850000000,7.52,7.460,14724,1,649398328.0,5.41,6.589,2.006017e+08,2.11,0.871
2,10,Star Wars Collection,Star Wars: The Force Awakens,2015-12-15,245000000,2068223624,8.44,7.250,20327,2,649398328.0,5.41,6.589,1.418825e+09,3.03,0.661
3,10,Star Wars Collection,Star Wars: The Last Jedi,2017-12-13,300000000,1332698830,4.44,6.757,16134,3,649398328.0,5.41,6.589,6.833005e+08,-0.97,0.168
4,10,Star Wars Collection,Star Wars: The Rise of Skywalker,2019-12-18,489900000,1074144248,2.19,6.272,10664,4,649398328.0,5.41,6.589,4.247459e+08,-3.22,-0.317
5,84,Indiana Jones Collection,Indiana Jones and the Kingdom of the Crystal S...,2008-05-21,185000000,786636033,4.25,6.019,8900,0,NaN,NaN,NaN,NaN,NaN,NaN
6,84,Indiana Jones Collection,Indiana Jones and the Dial of Destiny,2023-06-25,294700000,383963057,1.30,6.513,4028,1,786636033.0,4.25,6.019,-4.026730e+08,-2.95,0.494
7,119,The Lord of the Rings Collection,The Lord of the Rings: The Fellowship of the Ring,2001-12-18,93000000,871368364,9.37,8.429,27119,0,NaN,NaN,NaN,NaN,NaN,NaN
8,119,The Lord of the Rings Collection,The Lord of the Rings: The Two Towers,2002-12-18,79000000,926287400,11.73,8.413,23529,1,871368364.0,9.37,8.429,5.491904e+07,2.36,-0.016
9,119,The Lord of the Rings Collection,The Lord of the Rings: The Return of the King,2003-12-17,94000000,1118888979,11.90,8.494,26134,2,871368364.0,9.37,8.429,2.475206e+08,2.53,0.065


## Store

In [ ]:
df.to_csv("Q6.csv", index=False)
print("Gespeichert.")

Gespeichert.
